In [100]:
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate, FewShotPromptTemplate, PromptTemplate
from langchain_community.document_loaders import PyPDFLoader, TextLoader 
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

In [58]:
load_dotenv()

True

In [88]:
llm = ChatOpenAI(model='gpt-3.5-turbo')

In [59]:
## Steps
## Load Document, Embedding, Vector Store, Retrieval QA, LLM Chain

In [60]:
doccument = PyPDFLoader('Munna Biography.pdf').load()

In [61]:
embedding = OpenAIEmbeddings(model='text-embedding-3-large', dimensions=500)

In [62]:
splitter = RecursiveCharacterTextSplitter(chunk_size=600, separators=["Chapter", "\n\n", "\n", " ", ""])

In [63]:
splitted_doccuments = splitter.split_documents(doccument)

In [65]:
vectorstore = FAISS.from_documents(splitted_doccuments, embedding)

In [96]:
retriever = vectorstore.as_retriever(search_kwargs={"k":5,},search_type="similarity")

In [77]:
retriever.invoke("What is Munna's First Job?")

[Document(metadata={'source': 'Munna Biography.pdf', 'page': 0}, page_content='Chapter 3: First Steps in the Professional World\nMy first brush with earning independently came through tutoring students. Teaching sharpened my ability\nto break down complex ideas, explain concepts clearly, and cultivate patience — lessons that would later\nbecome invaluable in my professional journey.\nOn 14 March 2022, I officially stepped into the world of data science as a Junior Data Scientist at SSL\nWireless, one of Bangladesh’s largest fintech companies. It was a mixture of excitement and anxiety. I was'),
 Document(metadata={'source': 'Munna Biography.pdf', 'page': 2}, page_content='with ambition, and passion with purpose.\nMahmud Hasan Munna’s story is a testament to curiosity, resilience, and the power of self-directed learning.\nIt is a journey of a quiet mind that can create profound impact — a journey that is only just beginning.\n3')]

In [ ]:
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=retriever,
    llm=ChatOpenAI(model='gpt-3.5-turbo', temperature=0)
)

In [92]:
compression_retriver =  ContextualCompressionRetriever(
    base_compressor=LLMChainExtractor.from_llm(llm=llm),
    base_retriever=retriever
)

In [94]:
compression_retriver.invoke("Munna's First Job?")

[Document(metadata={'source': 'Munna Biography.pdf', 'page': 0}, page_content='Junior Data Scientist at SSL Wireless, one of Bangladesh’s largest fintech companies.')]

In [55]:
print(splitted_doccuments[1].page_content)

Chapter 1: Roots of Curiosity
I was born on 26 June 1998, in the small town of Mohanpur Upazila, Rajshahi, Bangladesh. My childhood
was filled with freedom, curiosity, and exploration. I was a child who wanted to understand how things
worked — not just follow instructions. I dismantled objects, tinkered with ideas, and often found myself lost
in thought, imagining possibilities.
My father , quietly strong and disciplined, influenced me more than anyone else. From him, I learned the
value of integrity, patience, and thoughtful action. My family was supportive, adventurous, and encouraging


In [127]:
chat_template = PromptTemplate(
    template="""Act like you are Munna. Answer the user query how Munna would, following the context. 
If you do not have sufficient context, say "I cannot answer that."

Context:
{context}

Question: {question}""",
    input_variables=['context', 'question']
)

In [121]:
llm = ChatOpenAI(model='gpt-5.1')


In [122]:
# chain = retriever| chat_template | llm

In [129]:
question = "what type of project did you work on?"
retrived_info = retriever.invoke(question)
context_text = "\n\n".join(doc.page_content for doc in retrived_info)
final_promt = chat_template.format(context=context_text, question=question)
print(llm.invoke(final_promt).content)

I’ve worked on a mix of data, ML, and analytics projects that sat right at the intersection of technology and business impact. Some key ones:

1. **Customer Profiling System (20M+ customers)**  
   - Designed a large-scale profiling system that became the backbone of targeted marketing.  
   - This directly contributed to generating **over 1 crore BDT in revenue** by enabling more precise, data-driven campaigns.

2. **eKYC System**  
   - Developed an electronic KYC (Know Your Customer) system to streamline customer onboarding.  
   - This reduced manual effort and operational overhead, **saving around 500,000 BDT**.

3. **Company-wide Data Warehouse**  
   - Built a centralized data warehouse for the organization.  
   - It integrated data from multiple sources and dramatically improved the **speed and efficiency of decision-making** across teams.

4. **ML Models, Data Pipelines, and Analytics at SSL Wireless**  
   - Worked on **machine learning models**, **data analysis**, and **end